<a href="https://colab.research.google.com/github/gracenaomi1122/my-first-repo/blob/main/Swiggy_workshop_17th_May_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛵 How Swiggy Tracks 10M Orders Daily
### A hands-on workshop with NumPy, Pandas & Google Colab

In [ ]:
# Run this once — it builds swiggy_orders.csv in the Colab session
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

rng = np.random.default_rng(42)
N = 3000

cities = ["Bengaluru", "Hyderabad", "Mumbai", "Delhi", "Chennai", "Pune", "Kolkata", "Ahmedabad"]
city_weights = [0.22, 0.16, 0.18, 0.15, 0.10, 0.08, 0.06, 0.05]
categories = ["South Indian", "North Indian", "Chinese", "Pizza", "Burgers", "Desserts", "Beverages", "Healthy Bowls"]
cat_weights = [0.18, 0.20, 0.14, 0.12, 0.10, 0.10, 0.08, 0.08]

restaurants = {
    "South Indian": ["Sri Udupi Palace", "Adyar Ananda Bhavan", "MTR", "Brahmin's Coffee Bar", "Dosa Plaza"],
    "North Indian": ["Sagar Ratna", "Haldiram's", "Rajdhani Thali", "Veg Treat", "Punjabi Tadka"],
    "Chinese": ["Wok in the Clouds", "Mainland Veg China", "Bowl Co.", "Chung Wah Veg"],
    "Pizza": ["Pizza Hut Veg", "La Pino'z", "Oven Story Veg", "Pizza Bakery"],
    "Burgers": ["Burger Singh Veg", "Veg Stop Burgers", "Plantman Burgers"],
    "Desserts": ["Theobroma", "Bakingo", "Ovenfresh", "Sweet Truth"],
    "Beverages": ["Chai Point", "Boba Bhai", "Blue Tokai", "Third Wave Coffee"],
    "Healthy Bowls": ["Salad Days", "The Bowl Company", "Greenr Cafe", "Eat.Fit"],
}
items_by_cat = {
    "South Indian": ["Masala Dosa", "Idli Sambar", "Vada", "Uttapam", "Filter Coffee", "Pongal", "Rava Dosa"],
    "North Indian": ["Paneer Butter Masala", "Dal Makhani", "Veg Biryani", "Chole Bhature", "Aloo Paratha", "Palak Paneer", "Mushroom Masala"],
    "Chinese": ["Veg Hakka Noodles", "Veg Manchurian", "Veg Fried Rice", "Schezwan Noodles", "Spring Rolls", "Chilli Paneer"],
    "Pizza": ["Margherita", "Farmhouse", "Paneer Tikka Pizza", "Veggie Supreme", "Cheese Burst Veg", "Mexican Green Wave"],
    "Burgers": ["Aloo Tikki Burger", "Paneer Burger", "Veg Whopper", "Crispy Veg Burger"],
    "Desserts": ["Choco Lava Cake", "Gulab Jamun", "Brownie", "Rasmalai", "Tiramisu (eggless)", "Cheesecake (eggless)"],
    "Beverages": ["Masala Chai", "Cold Coffee", "Mango Smoothie", "Lemonade", "Hot Chocolate"],
    "Healthy Bowls": ["Quinoa Bowl", "Buddha Bowl", "Falafel Bowl", "Khichdi Bowl", "Veg Poke Bowl"],
}
price_range = {"South Indian":(80,250),"North Indian":(150,450),"Chinese":(140,380),"Pizza":(200,600),
               "Burgers":(120,320),"Desserts":(100,350),"Beverages":(60,220),"Healthy Bowls":(180,420)}
payment_modes = ["UPI","Card","Cash","Wallet"]; pay_weights=[0.55,0.22,0.13,0.10]
statuses = ["Delivered","Cancelled"]; status_w=[0.93,0.07]

rows = []
base_time = datetime(2024, 11, 1)
for i in range(N):
    city = rng.choice(cities, p=city_weights)
    category = rng.choice(categories, p=cat_weights)
    restaurant = rng.choice(restaurants[category])
    item = rng.choice(items_by_cat[category])
    lo, hi = price_range[category]
    item_price = float(rng.integers(lo, hi+1))
    quantity = int(rng.choice([1,1,1,2,2,3], p=[0.45,0.10,0.05,0.25,0.10,0.05]))
    delivery_fee = float(rng.integers(20,60)) + (10 if city in ("Mumbai","Bengaluru") else 0)
    discount = float(rng.choice([0,0,0,20,30,50,75,100], p=[0.45,0.10,0.05,0.10,0.10,0.10,0.06,0.04]))
    base_d = {"Bengaluru":38,"Hyderabad":32,"Mumbai":42,"Delhi":36,"Chennai":30,"Pune":33,"Kolkata":34,"Ahmedabad":29}[city]
    delivery_time = int(np.clip(rng.normal(base_d, 8), 12, 95))
    rating = float(np.round(np.clip(rng.normal(4.2, 0.6), 1.0, 5.0), 1))
    status = rng.choice(statuses, p=status_w)
    payment_mode = rng.choice(payment_modes, p=pay_weights)
    order_time = base_time + timedelta(days=int(rng.integers(0,30)), hours=int(rng.integers(8,24)), minutes=int(rng.integers(0,60)))
    rows.append({"order_id":100000+i,"order_time":order_time,"city":city,"restaurant":restaurant,
                 "category":category,"item":item,"item_price":item_price,"quantity":quantity,
                 "delivery_fee":delivery_fee,"discount":discount,"payment_mode":payment_mode,
                 "delivery_time_min":delivery_time,"rating":rating,"status":status})

df_raw = pd.DataFrame(rows)

# Inject realistic mess
null_idx = rng.choice(df_raw.index, size=int(0.15*len(df_raw)), replace=False)
df_raw.loc[null_idx, "rating"] = np.nan
null_idx2 = rng.choice(df_raw.index, size=int(0.03*len(df_raw)), replace=False)
df_raw.loc[null_idx2, "delivery_time_min"] = np.nan
null_idx3 = rng.choice(df_raw.index, size=int(0.02*len(df_raw)), replace=False)
df_raw.loc[null_idx3, "payment_mode"] = np.nan
cancelled = df_raw["status"] == "Cancelled"
df_raw.loc[cancelled & (rng.random(len(df_raw)) < 0.7), "delivery_time_min"] = np.nan
df_raw.loc[cancelled & (rng.random(len(df_raw)) < 0.8), "rating"] = np.nan

# Duplicates + casing typos
dup_sample = df_raw.sample(n=25, random_state=7)
df_raw = pd.concat([df_raw, dup_sample], ignore_index=True)
typo_idx = rng.choice(df_raw.index, size=15, replace=False)
df_raw.loc[typo_idx, "city"] = df_raw.loc[typo_idx, "city"].str.lower()
df_raw = df_raw.sample(frac=1.0, random_state=11).reset_index(drop=True)

df_raw.to_csv("swiggy_orders.csv", index=False)
print(f"✅ Dataset created: swiggy_orders.csv ({len(df_raw)} rows)")

✅ Dataset created: swiggy_orders.csv (3025 rows)
